# Collecting Dataset

In [1]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Create a shortcut to the Gallery and Probe Set Folders to your MyDrive (or change the path)
gallery_path = '/content/drive/Shareddrives/PhotoDS3001/ClassGallerySet'
# probe_path = '/content/drive/MyDrive/ProbeSet'

# Arcface

**install** insighface package for google colab & pillow-heif for handling HEIF files

In [3]:
!pip install -U insightface

import torch
if torch.cuda.is_available():
  print('gpu is available')
  !pip install onnxruntime-gpu  # to use GPU
else:
  print('gpu is not available')
  !pip install onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 14.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 96.2 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071486 sha256=f71c9df5b21aa2cbaf102f4e3e74aebe699bb27b78c62a33868030ed04c5b6d4
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface
gpu is available
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 4.8 MB/s eta 0:00:00


In [4]:
!pip install pillow-heif
from pillow_heif import register_heif_opener
register_heif_opener()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 52.0 MB/s eta 0:00:00


**prepare insightface module**

In [5]:
import insightface
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image

app = FaceAnalysis()
app.prepare(ctx_id=0, det_thresh=0.5)

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:05<00:00, 54704.13KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

**ONYX graph inspection**

In [6]:
import onnx
model = onnx.load('/root/.insightface/models/buffalo_l/w600k_r50.onnx')
print(onnx.helper.printable_graph(model.graph))

graph torch-jit-export (
  %input.1[FLOAT, Nonex3x112x112]
) initializers (
  %layer1.0.bn1.weight[FLOAT, 64]
  %layer1.0.bn1.bias[FLOAT, 64]
  %layer1.0.bn1.running_mean[FLOAT, 64]
  %layer1.0.bn1.running_var[FLOAT, 64]
  %layer1.1.bn1.weight[FLOAT, 64]
  %layer1.1.bn1.bias[FLOAT, 64]
  %layer1.1.bn1.running_mean[FLOAT, 64]
  %layer1.1.bn1.running_var[FLOAT, 64]
  %layer1.2.bn1.weight[FLOAT, 64]
  %layer1.2.bn1.bias[FLOAT, 64]
  %layer1.2.bn1.running_mean[FLOAT, 64]
  %layer1.2.bn1.running_var[FLOAT, 64]
  %layer2.0.bn1.weight[FLOAT, 64]
  %layer2.0.bn1.bias[FLOAT, 64]
  %layer2.0.bn1.running_mean[FLOAT, 64]
  %layer2.0.bn1.running_var[FLOAT, 64]
  %layer2.1.bn1.weight[FLOAT, 128]
  %layer2.1.bn1.bias[FLOAT, 128]
  %layer2.1.bn1.running_mean[FLOAT, 128]
  %layer2.1.bn1.running_var[FLOAT, 128]
  %layer2.2.bn1.weight[FLOAT, 128]
  %layer2.2.bn1.bias[FLOAT, 128]
  %layer2.2.bn1.running_mean[FLOAT, 128]
  %layer2.2.bn1.running_var[FLOAT, 128]
  %layer2.3.bn1.weight[FLOAT, 128]
  %layer2.3

/tmp/ipykernel_5125/3736105999.py:3: DeprecationWarning: Deprecated since 1.19. Consider using onnx.printer.to_text() instead.
  print(onnx.helper.printable_graph(model.graph))


**feature extraction**

In [7]:
import os
import cv2
import numpy as np
from pillow_heif import register_heif_opener
from PIL import Image
register_heif_opener()

gallery_path = '/content/drive/MyDrive/PhotoDS3001/ClassGallerySet'

def extract_arcface_features(dataset_path):
  features = []
  labels = []
  filenames = []

  for person in sorted(os.listdir(dataset_path)):
    person_path = os.path.join(dataset_path, person)
    if not os.path.isdir(person_path):
      continue
    for file in sorted(os.listdir(person_path)):
      if not file.lower().endswith(('.jpg', '.jpeg', '.png', '.heic')):
        continue
      filepath = os.path.join(person_path, file)
      try:
        # Handle HEIC and regular images
        pil_img = Image.open(filepath).convert('RGB')
        img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        faces = app.get(img)
        if len(faces) == 0:
          print(f"  No face detected: {person}/{file}")
          continue
        if len(faces) > 1:
          print(f"  Warning: {len(faces)} faces detected in {person}/{file}, using largest")
        features.append(faces[0].normed_embedding)
        labels.append(person)
        filenames.append(file)
        print(f"  processed: {person}/{file}")
      except Exception as e:
        print(f"  Error on {person}/{file}: {e}")
  return np.array(features), labels, filenames

print("Extracting gallery features...")
arc_gallery_features, arc_gallery_labels, arc_gallery_files = extract_arcface_features(gallery_path)
print(f"\nGallery: {len(arc_gallery_features)} features extracted")

# print("\nExtracting probe features...")
# arc_probe_features, arc_probe_labels, arc_probe_files = extract_arcface_features(probe_path)
# print(f"\nProbe: {len(arc_probe_features)} features extracted")

Extracting gallery features...
  processed: Abigail Crow/IMG_6343.HEIC
  processed: Abigail Crow/IMG_6344.HEIC
  processed: Abigail Crow/IMG_6345.HEIC
  processed: Abigail Crow/IMG_6346.HEIC
  processed: Abigail Crow/IMG_6347.HEIC
  processed: Abigail Crow/IMG_6348.HEIC
  processed: Abigail Crow/IMG_6349.HEIC
  processed: Abigail Crow/IMG_6350.HEIC
  processed: Abigail Crow/IMG_6351.HEIC
  processed: Abigail Crow/IMG_6352.HEIC
  processed: Ally O’Shea/IMG_6292.HEIC
  processed: Ally O’Shea/IMG_6293.HEIC
  processed: Ally O’Shea/IMG_6294.HEIC
  processed: Ally O’Shea/IMG_6295.HEIC
  processed: Ally O’Shea/IMG_6296.HEIC
  processed: Ally O’Shea/IMG_6297.HEIC
  processed: Ally O’Shea/IMG_6298.HEIC
  processed: Ally O’Shea/IMG_6299.HEIC
  processed: Ally O’Shea/IMG_6300.HEIC
  processed: Ally O’Shea/IMG_6301.HEIC
  processed: Ananya Balachander/IMG_6369.HEIC
  processed: Ananya Balachander/IMG_6370.HEIC
  processed: Ananya Balachander/IMG_6371.HEIC
  processed: Ananya Balachander/IMG_6372.

# VGG

**install and prepare module**

In [8]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

vgg = models.vgg19(weights='DEFAULT')

feature_extractor = torch.nn.Sequential(
    vgg.features,
    vgg.avgpool,
    torch.nn.Flatten(),
    vgg.classifier[0],  # Linear 25088 --> 4096
    vgg.classifier[1],  # ReLU
).to(device)
feature_extractor.eval()

Using device: cuda
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:09<00:00, 58.8MB/s]


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [9]:
# vgg normally

# import os
# import numpy as np
# from PIL import Image

# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

# def extract_vgg_features(dataset_path):
#   features = []
#   labels = []
#   filenames = []

#   for person in sorted(os.listdir(dataset_path)):
#     person_path = os.path.join(dataset_path, person)
#     if not os.path.isdir(person_path):
#       continue
#     for file in sorted(os.listdir(person_path)):
#       if not file.lower().endswith(('.jpg', '.jpeg', '.png', '.heic')):
#         continue
#       filepath = os.path.join(person_path, file)
#       try:
#         img = Image.open(filepath).convert('RGB')
#         tensor = transform(img).unsqueeze(0).to(device)  # send to GPU
#         with torch.no_grad():
#           feat = feature_extractor(tensor)
#         features.append(feat.squeeze().cpu().numpy())  # back to CPU
#         labels.append(person)
#         filenames.append(file)
#         print(f"  processed: {person}/{file}")
#       except Exception as e:
#         print(f"  Error on {person}/{file}: {e}")

#   return np.array(features), labels, filenames

# print("Extracting VGG gallery features...")
# vgg_gallery_features, vgg_gallery_labels, vgg_gallery_files = extract_vgg_features(gallery_path)
# print(f"\nGallery: {len(vgg_gallery_features)} features extracted")

# # print("\nExtracting VGG probe features...")
# # vgg_probe_features, vgg_probe_labels, vgg_probe_files = extract_vgg_features(probe_path)
# # print(f"\nProbe: {len(vgg_probe_features)} features extracted")

Extracting VGG gallery features...
  processed: Abigail Crow/IMG_6343.HEIC
  processed: Abigail Crow/IMG_6344.HEIC
  processed: Abigail Crow/IMG_6345.HEIC
  processed: Abigail Crow/IMG_6346.HEIC
  processed: Abigail Crow/IMG_6347.HEIC
  processed: Abigail Crow/IMG_6348.HEIC
  processed: Abigail Crow/IMG_6349.HEIC
  processed: Abigail Crow/IMG_6350.HEIC
  processed: Abigail Crow/IMG_6351.HEIC
  processed: Abigail Crow/IMG_6352.HEIC
  processed: Ally O’Shea/IMG_6292.HEIC
  processed: Ally O’Shea/IMG_6293.HEIC
  processed: Ally O’Shea/IMG_6294.HEIC
  processed: Ally O’Shea/IMG_6295.HEIC
  processed: Ally O’Shea/IMG_6296.HEIC
  processed: Ally O’Shea/IMG_6297.HEIC
  processed: Ally O’Shea/IMG_6298.HEIC
  processed: Ally O’Shea/IMG_6299.HEIC
  processed: Ally O’Shea/IMG_6300.HEIC
  processed: Ally O’Shea/IMG_6301.HEIC
  processed: Ananya Balachander/IMG_6369.HEIC
  processed: Ananya Balachander/IMG_6370.HEIC
  processed: Ananya Balachander/IMG_6371.HEIC
  processed: Ananya Balachander/IMG_6

# vgg as 1000-dim

In [10]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vgg_full = models.vgg16(pretrained=True)
vgg_full.eval()
vgg_full.to(device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def extract_vgg1000_features(dataset_path):
    features = []
    labels = []

    for person in sorted(os.listdir(dataset_path)):
        person_path = os.path.join(dataset_path, person)
        if not os.path.isdir(person_path):
            continue
        for file in sorted(os.listdir(person_path)):
            if not file.lower().endswith(('.jpg', '.jpeg', '.png', '.heic')):
                continue
            filepath = os.path.join(person_path, file)
            try:
                img = Image.open(filepath).convert('RGB')
                tensor = transform(img).unsqueeze(0).to(device)
                with torch.no_grad():
                    feat = vgg_full(tensor)          # full forward pass → 1000-dim
                features.append(feat.squeeze().cpu().numpy())
                labels.append(person)
                print(f'  processed: {person}/{file}')
            except Exception as e:
                print(f'  Error on {person}/{file}: {e}')

    return np.array(features), labels

print('Extracting 1000-dim VGG gallery features...')
vgg1000_features, vgg1000_labels = extract_vgg1000_features(gallery_path)
print(f'\nDone! Gallery: {vgg1000_features.shape}')

# Save it
save_path = '/content/drive/MyDrive/'
np.save(save_path + 'vgg1000_gallery_class_features.npy', vgg1000_features)
np.save(save_path + 'vgg1000_gallery_class_labels.npy', np.array(vgg1000_labels))
print('Saved to Drive!')

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 123MB/s] 


Extracting 1000-dim VGG gallery features...
  processed: Abigail Crow/IMG_6343.HEIC
  processed: Abigail Crow/IMG_6344.HEIC
  processed: Abigail Crow/IMG_6345.HEIC
  processed: Abigail Crow/IMG_6346.HEIC
  processed: Abigail Crow/IMG_6347.HEIC
  processed: Abigail Crow/IMG_6348.HEIC
  processed: Abigail Crow/IMG_6349.HEIC
  processed: Abigail Crow/IMG_6350.HEIC
  processed: Abigail Crow/IMG_6351.HEIC
  processed: Abigail Crow/IMG_6352.HEIC
  processed: Ally O’Shea/IMG_6292.HEIC
  processed: Ally O’Shea/IMG_6293.HEIC
  processed: Ally O’Shea/IMG_6294.HEIC
  processed: Ally O’Shea/IMG_6295.HEIC
  processed: Ally O’Shea/IMG_6296.HEIC
  processed: Ally O’Shea/IMG_6297.HEIC
  processed: Ally O’Shea/IMG_6298.HEIC
  processed: Ally O’Shea/IMG_6299.HEIC
  processed: Ally O’Shea/IMG_6300.HEIC
  processed: Ally O’Shea/IMG_6301.HEIC
  processed: Ananya Balachander/IMG_6369.HEIC
  processed: Ananya Balachander/IMG_6370.HEIC
  processed: Ananya Balachander/IMG_6371.HEIC
  processed: Ananya Balachan

# Downloading Extracted Files

In [11]:
import numpy as np

save_path = '/content/drive/MyDrive/'

# Save Arcface features
np.save(save_path + 'arcface_gallery_class_features.npy', arc_gallery_features)
#np.save(save_path + 'arcface_probe_features.npy', arc_probe_features)
np.save(save_path + 'arcface_gallery_class_labels.npy', np.array(arc_gallery_labels))
#np.save(save_path + 'arcface_probe_labels.npy', np.array(arc_probe_labels))

print("All Arcface features saved to Drive!")
print(f"Arcface gallery: {arc_gallery_features.shape}")
#rint(f"Arcface probe: {arc_probe_features.shape}")

# Save VGG features
np.save(save_path + 'vgg_gallery_class_features.npy', vgg_gallery_features)
#np.save(save_path + 'vgg_probe_features.npy', vgg_probe_features)
np.save(save_path + 'vgg_gallery_class_labels.npy', np.array(vgg_gallery_labels))
#np.save(save_path + 'vgg_probe_labels.npy', np.array(vgg_probe_labels))

print("All VGG features saved to Drive!")
print(f"VGG gallery: {vgg_gallery_features.shape}")
#print(f"VGG probe: {vgg_probe_features.shape}")

All Arcface features saved to Drive!
Arcface gallery: (261, 512)
All VGG features saved to Drive!
VGG gallery: (261, 4096)


In [16]:
import numpy as np
from scipy.spatial.distance import cosine
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt


GALLERY_PATH = '/content/drive/MyDrive/'
MYSTERY_PATH = '/content/drive/MyDrive/Mystery/'

arc_feats  = np.load(GALLERY_PATH + 'arcface_gallery_class_features.npy')
arc_labels = np.load(GALLERY_PATH + 'arcface_gallery_class_labels.npy', allow_pickle=True)
vgg_feats  = np.load(GALLERY_PATH + 'vgg1000_gallery_class_features.npy')
vgg_labels = np.load(GALLERY_PATH + 'vgg1000_gallery_class_labels.npy', allow_pickle=True)

print(f'ArcFace gallery : {arc_feats.shape}')
print(f'VGG gallery     : {vgg_feats.shape}')


def load_mystery(path):
    raw = np.load(path, allow_pickle=True)
    if raw.shape == () and isinstance(raw.item(), dict):
        vec = np.array(list(raw.item().values())[0]).flatten()
    elif raw.ndim == 2 and raw.shape[0] == 1:
        vec = raw.flatten()
    else:
        vec = raw.flatten()
    return vec.astype(np.float32)


def cosine_sim(a, b):
    return 1.0 - cosine(a, b)


def match(vec, gallery_feats, gallery_labels):
    best = defaultdict(float)
    for feat, label in zip(gallery_feats, gallery_labels):
        sim = cosine_sim(vec, feat)
        if sim > best[label]:
            best[label] = sim
    return sorted(best.items(), key=lambda x: x[1], reverse=True)

ID_MODEL = {
    'ID1': 'arcface',
    'ID2': 'arcface',
    'ID3': 'arcface',
    'ID4': 'vgg',
    'ID5': 'vgg',
}

results = []

for mystery_id, model in ID_MODEL.items():
    path = MYSTERY_PATH + f'{mystery_id}.npy'
    vec  = load_mystery(path)

    if model == 'arcface':
        if vec.shape[0] != arc_feats.shape[1]:
            print(f'{mystery_id}: expected {arc_feats.shape[1]}-dim ArcFace but got {vec.shape[0]}. Skipping.')
            continue
        ranked = match(vec, arc_feats, arc_labels)
    else:
        if vec.shape[0] != vgg_feats.shape[1]:
            print(f' {mystery_id}: expected {vgg_feats.shape[1]}-dim VGG but got {vec.shape[0]}. Skipping.')
            continue
        ranked = match(vec, vgg_feats, vgg_labels)

    best_person, best_sim = ranked[0]
    print(f'\n{mystery_id}  [{model.upper()}]')
    print(f'BEST MATCH : {best_person:<30s}  similarity = {best_sim:.4f}')
    for rank, (person, sim) in enumerate(ranked[1:3], start=2):
        print(f'   #{rank} runner-up : {person:<30s}  similarity = {sim:.4f}')

    results.append({
        'Mystery ID': mystery_id,
        'Model': model,
        'Best Match': best_person,
        'Similarity': round(float(best_sim), 4),
        '2nd Place': ranked[1][0] if len(ranked) > 1 else '',
        '3rd Place': ranked[2][0] if len(ranked) > 2 else '',
    })

df = pd.DataFrame(results)
print(df[['Mystery ID', 'Model', 'Best Match', 'Similarity']].to_string(index=False))

ArcFace gallery : (261, 512)
VGG gallery     : (261, 1000)

ID1  [ARCFACE]
BEST MATCH : Nicolas Ruiz Rodriguez          similarity = 0.2673
   #2 runner-up : Sunny Park                      similarity = 0.2650
   #3 runner-up : Dylan Mulieri                   similarity = 0.2210

ID2  [ARCFACE]
BEST MATCH : Kiersten Bruns                  similarity = 0.6819
   #2 runner-up : Kerry Xiao                      similarity = 0.4132
   #3 runner-up : Lily Williams                   similarity = 0.2330

ID3  [ARCFACE]
BEST MATCH : Kerry Xiao                      similarity = 0.5502
   #2 runner-up : Nicolas Ruiz Rodriguez          similarity = 0.1390
   #3 runner-up : Ciara Graves                    similarity = 0.1182

ID4  [VGG]
BEST MATCH : Ciara Graves                    similarity = 0.7031
   #2 runner-up : Lily Williams                   similarity = 0.6855
   #3 runner-up : Jaitha Jasti                    similarity = 0.6850

ID5  [VGG]
BEST MATCH : Marlise Lucas                   simi